# RAG — Évaluation d'indexation

Ce notebook permet de tester l'indexation d'un fichier PDF dans Qdrant et de mesurer :
- Le nombre de chunks générés
- Le nombre de tokens consommés
- Le coût estimé (OpenAI `text-embedding-3-small`)
- Le temps d'indexation
- La qualité de la recherche vectorielle

**Prérequis :**
```bash
pip install openai qdrant-client pypdf tiktoken
# Qdrant doit tourner localement :
# docker compose up qdrant
```

In [ ]:
import os
import time
import uuid
from pathlib import Path

import tiktoken
import pypdf
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, FilterSelector

# ── Configuration ─────────────────────────────────────────────────────────
OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY", "")  # ou coller la clé directement
QDRANT_URL       = "http://localhost:6333"
COLLECTION_NAME  = "rd_rag_test"                     # collection isolée pour les tests
EMBEDDING_MODEL  = "text-embedding-3-small"
EMBEDDING_DIM    = 1536
CHUNK_SIZE       = 600    # en caractères (même valeur que la prod)
CHUNK_OVERLAP    = 100

# Coût OpenAI text-embedding-3-small : 0.02$ / 1M tokens
COST_PER_1M_TOKENS = 0.02

print("✓ Imports OK")

## 1. Choisir le fichier à tester

In [ ]:
# ── Modifier ce chemin pour tester un autre fichier ───────────────────────
FILE_PATH = Path("../../knowledge_base/templates/note_methodologique_ONSSA.pdf")

assert FILE_PATH.exists(), f"Fichier introuvable : {FILE_PATH.resolve()}"
size_kb = FILE_PATH.stat().st_size / 1024
print(f"Fichier : {FILE_PATH.name}")
print(f"Taille  : {size_kb:.1f} KB")

## 2. Extraction du texte

In [ ]:
def extract_pdf(path: Path) -> str:
    reader = pypdf.PdfReader(str(path))
    pages = [p.extract_text() or "" for p in reader.pages]
    return "\n".join(pages)

t0 = time.time()
text = extract_pdf(FILE_PATH)
t_extract = time.time() - t0

print(f"Pages    : {len(pypdf.PdfReader(str(FILE_PATH)).pages)}")
print(f"Caractères extraits : {len(text):,}")
print(f"Temps extraction    : {t_extract:.2f}s")
print()
print("--- Aperçu (500 premiers caractères) ---")
print(text[:500])

## 3. Chunking

In [ ]:
def chunk_text(text: str, size: int, overlap: int) -> list[str]:
    text = text.strip()
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            cut = text.rfind(" ", start, end)
            if cut > start:
                end = cut
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        start = end - overlap
    return chunks

chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"Nombre de chunks    : {len(chunks)}")
print(f"Taille moyenne      : {sum(len(c) for c in chunks) / len(chunks):.0f} caractères")
print(f"Plus petit chunk    : {min(len(c) for c in chunks)} caractères")
print(f"Plus grand chunk    : {max(len(c) for c in chunks)} caractères")
print()
print("--- Exemple chunk #1 ---")
print(chunks[0])

## 4. Comptage des tokens et estimation du coût

In [ ]:
enc = tiktoken.encoding_for_model("text-embedding-3-small")

token_counts = [len(enc.encode(c)) for c in chunks]
total_tokens = sum(token_counts)
cost_usd     = (total_tokens / 1_000_000) * COST_PER_1M_TOKENS
cost_eur     = cost_usd * 0.92  # approximation

print(f"Tokens totaux       : {total_tokens:,}")
print(f"Tokens moyens/chunk : {total_tokens / len(chunks):.0f}")
print(f"Coût estimé         : ${cost_usd:.6f} USD  (~{cost_eur*100:.4f} centimes EUR)")
print()

# Extrapolation pour toute la knowledge_base
print("--- Extrapolation knowledge_base ---")
nb_docs_kb = 20  # nombre estimé de documents
cost_kb = cost_eur * nb_docs_kb
print(f"Si {nb_docs_kb} documents similaires : ~{cost_kb*100:.2f} centimes EUR au total")

## 5. Génération des embeddings

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

t0 = time.time()
response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=chunks,
)
t_embed = time.time() - t0

embeddings = [d.embedding for d in response.data]
tokens_used = response.usage.total_tokens

print(f"Embeddings générés  : {len(embeddings)}")
print(f"Tokens facturés     : {tokens_used:,}")
print(f"Dimension vecteur   : {len(embeddings[0])}")
print(f"Temps embedding     : {t_embed:.2f}s")
print(f"Coût réel           : ${(tokens_used / 1_000_000) * COST_PER_1M_TOKENS:.6f} USD")

## 6. Stockage dans Qdrant

In [ ]:
qdrant = QdrantClient(url=QDRANT_URL)

# Créer une collection de test isolée
existing = {c.name for c in qdrant.get_collections().collections}
if COLLECTION_NAME in existing:
    qdrant.delete_collection(COLLECTION_NAME)
    print(f"Collection '{COLLECTION_NAME}' réinitialisée")

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

# Insérer les points
t0 = time.time()
points = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=emb,
        payload={
            "content":   chunk,
            "doc_name":  FILE_PATH.name,
            "chunk_idx": idx,
        },
    )
    for idx, (chunk, emb) in enumerate(zip(chunks, embeddings))
]
qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
t_store = time.time() - t0

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Points stockés      : {info.points_count}")
print(f"Temps stockage      : {t_store:.2f}s")

## 7. Test de recherche

In [ ]:
# ── Modifier la requête pour tester la pertinence ─────────────────────────
QUERY = "méthodologie approche formation"
TOP_K = 3

t0 = time.time()
query_emb = client.embeddings.create(model=EMBEDDING_MODEL, input=[QUERY]).data[0].embedding
results   = qdrant.search(collection_name=COLLECTION_NAME, query_vector=query_emb, limit=TOP_K)
t_search  = time.time() - t0

print(f"Requête : '{QUERY}'")
print(f"Temps recherche : {t_search*1000:.0f}ms")
print()
for i, r in enumerate(results, 1):
    print(f"--- Résultat #{i} (score: {r.score:.4f}) ---")
    print(r.payload["content"][:300])
    print()

In [ ]:
import json as _json

# ── Configuration ────────────────────────────────────────────────────────
N_CANDIDATES  = 20   # candidats Qdrant (filet large)
RERANK_TOP_K  = 3    # résultats finaux après reranking
RERANK_MODEL  = "gpt-4o-mini"

RERANK_SYSTEM = (
    "You are a relevance scoring assistant. "
    "Given a query and a list of text chunks, identify the most relevant ones. "
    "Return ONLY a JSON array of integers (chunk indices), nothing else."
)

RERANK_USER = """\
Query: {query}

Chunks:
{chunks_text}

Return a JSON array of the {top_k} most relevant chunk indices (0-based), \
sorted by relevance (most relevant first).
Example: [5, 2, 11]
Return ONLY the JSON array."""

# ── Étape 1 : Qdrant top-20 ──────────────────────────────────────────────
t0 = time.time()
candidates = qdrant.search(
    collection_name=COLLECTION_NAME,
    query_vector=query_emb,   # vecteur calculé en cellule 7
    limit=N_CANDIDATES,
    with_payload=True,
)
t_qdrant_wide = time.time() - t0

print(f"Qdrant top-{N_CANDIDATES} récupérés en {t_qdrant_wide*1000:.0f}ms")
print()
print("── Top-3 AVANT reranking (similarité cosinus) ──────────────────────")
for i, r in enumerate(candidates[:RERANK_TOP_K], 1):
    print(f"  #{i} score={r.score:.4f} | {r.payload['content'][:120].strip()!r}")
print()

# ── Étape 2 : Reranking via LLM cheap ───────────────────────────────────
chunks_text = "\n".join(
    f"[{i}] {r.payload.get('content', '')[:400]}"
    for i, r in enumerate(candidates)
)
user_prompt = RERANK_USER.format(
    query=QUERY,
    chunks_text=chunks_text,
    top_k=RERANK_TOP_K,
)

t0 = time.time()
rerank_resp = client.chat.completions.create(
    model=RERANK_MODEL,
    messages=[
        {"role": "system", "content": RERANK_SYSTEM},
        {"role": "user",   "content": user_prompt},
    ],
    max_tokens=50,
    temperature=0,
)
t_rerank = time.time() - t0

raw = rerank_resp.choices[0].message.content.strip()
print(f"Réponse LLM brute : {raw!r}")
print(f"Temps reranking   : {t_rerank*1000:.0f}ms")
print(f"Tokens reranker   : {rerank_resp.usage.total_tokens}")
print()

# ── Étape 3 : Validation et résultats rerankés ───────────────────────────
try:
    indices = _json.loads(raw)
    valid   = [i for i in indices if isinstance(i, int) and 0 <= i < len(candidates)]
    reranked = [candidates[i] for i in valid[:RERANK_TOP_K]]
    if len(reranked) < RERANK_TOP_K:
        seen = set(valid[:RERANK_TOP_K])
        for i, c in enumerate(candidates):
            if i not in seen:
                reranked.append(c)
            if len(reranked) >= RERANK_TOP_K:
                break
    print(f"Indices sélectionnés par le reranker : {valid[:RERANK_TOP_K]}")
    print()
    print("── Top-3 APRÈS reranking ────────────────────────────────────────────")
    for i, r in enumerate(reranked, 1):
        orig_idx = candidates.index(r)
        orig_score = r.score
        print(f"  #{i} (était #{orig_idx+1} Qdrant, score={orig_score:.4f}) | {r.payload['content'][:120].strip()!r}")
except Exception as e:
    print(f"⚠ Erreur parsing reranker : {e} — fallback top-{RERANK_TOP_K} Qdrant")
    reranked = candidates[:RERANK_TOP_K]

# ── Bilan ────────────────────────────────────────────────────────────────
print()
print("═" * 60)
print("BILAN RERANKER")
print("═" * 60)
print(f"Qdrant top-{N_CANDIDATES}         : {t_qdrant_wide*1000:.0f}ms")
print(f"LLM reranking (gpt-4o-mini) : {t_rerank*1000:.0f}ms")
print(f"Total                       : {(t_qdrant_wide + t_rerank)*1000:.0f}ms")
cost_rerank = (rerank_resp.usage.total_tokens / 1_000_000) * 0.15 * 0.92
print(f"Coût reranker               : {cost_rerank*100:.5f} centimes EUR")
print(f"Coût × 8 sections           : {cost_rerank*8*100:.4f} centimes EUR")
print("═" * 60)

## 7b. Test du Reranker

Simule la passe de reranking : on récupère **top-20** candidats Qdrant, puis un LLM cheap (`gpt-4o-mini`) re-classe et sélectionne les **top-3** vraiment pertinents.

On compare ensuite :
- **Avant** : top-3 bruts Qdrant (similarité cosinus)
- **Après** : top-3 rerankés (pertinence sémantique réelle)

## 8. Récapitulatif des métriques

In [ ]:
print("═" * 50)
print("RÉCAPITULATIF")
print("═" * 50)
print(f"Fichier             : {FILE_PATH.name}")
print(f"Taille              : {size_kb:.1f} KB")
print(f"Chunks              : {len(chunks)}")
print(f"Tokens              : {tokens_used:,}")
print()
print("TEMPS")
print(f"  Extraction PDF    : {t_extract:.2f}s")
print(f"  Embedding OpenAI  : {t_embed:.2f}s")
print(f"  Stockage Qdrant   : {t_store:.2f}s")
print(f"  Total indexation  : {t_extract + t_embed + t_store:.2f}s")
print(f"  Recherche         : {t_search*1000:.0f}ms")
print()
print("COÛT")
real_cost_eur = (tokens_used / 1_000_000) * COST_PER_1M_TOKENS * 0.92
print(f"  Ce fichier        : {real_cost_eur*100:.5f} centimes EUR")
print(f"  20 fichiers       : {real_cost_eur * 20 * 100:.3f} centimes EUR")
print(f"  100 fichiers      : {real_cost_eur * 100 * 100:.2f} centimes EUR")
print("═" * 50)

## 9. Nettoyage (optionnel)

In [ ]:
# Décommenter pour supprimer la collection de test
# qdrant.delete_collection(COLLECTION_NAME)
# print(f"Collection '{COLLECTION_NAME}' supprimée")